# Game 28 science-win audit

## tl;dr
Won at Modern turn 93 after reaching 118 Innovation at turn 88. The six retained cities were enough to win. The largest observed inefficiency was sustained local unhappiness, aggravated by policy and specialist choices. Rocketry opened at M51; three cities completed Preparation before the countdown, despite five completed Launch Pads.

The detailed findings and experimental slider schedule are in `../34-game28-analysis.md`.

## Context & Methods
This is a diagnostic companion to the consolidated findings list requested in the chat. Inputs are preserved in `evidence.zip`; the helper contains the extraction and calculation code. No bot settings are changed.

### Key Assumptions
- Turn numbers reset at age changes; use age + turn as the time key.
- Hall of Fame city series contain change points. Carry values forward only until a null end marker.
- Empire statistics and local city yields are different measures; do not interchange them.
- Happiness recovery is a fixed-state upper bound. It excludes the cost of obtaining happiness and is not a predicted improvement in win turn.
- Building alternatives were not logged. Recommendations are experimental, not an optimized replay.


## Data
### 1. Load and validate the preserved sources

In [1]:
from pathlib import Path
import sys, json
folder = Path.cwd()
if not (folder / "evidence.zip").exists():
    folder = folder / "research" / "game28"
sys.path.insert(0, str(folder))
import analyze
raw_tables, ui, game_core, provenance = analyze.load()
summary, stats, builds, frames, runs = analyze.summarize(raw_tables, ui, game_core)
history, city_turns = analyze.city_history()
assert summary["quality"]["stats_duplicate_keys"] == 0
assert [summary["ages"][age]["last"]["turn"] for age in analyze.AGES] == [70,70,93]
assert "Team 0 has won" in game_core and "VICTORY_SCIENCE_MODERN" in game_core
assert history["game"]["GameID"] == 30
assert len({(r["ObjectID"],r["DataSet"],r["turn"]) for r in history["rows"]}) == len(history["rows"])
print(json.dumps(summary["quality"], indent=2))
print("Snapshot source:", provenance["UI.log"]["source"])
print("UI SHA-256:", provenance["UI.log"]["sha256"])
print("Hall of Fame game:", history["game"]["GameID"], history["game"]["GameSpeedType"])


{
  "stats_rows": 4332,
  "stats_duplicate_keys": 0,
  "stats_header_columns": 22,
  "stats_row_columns": [
    25
  ],
  "ui_frames": 232,
  "unassigned_ui_lines": 1
}
Snapshot source: C:\Users\glazu\AppData\Local\Firaxis Games\Sid Meier's Civilization VII\Logs\UI.log
UI SHA-256: 7bcaf515298c71bf9fae9dd7671dac0db7185cbac25d3b03da63c17e6fd5621c
Hall of Fame game: 30 GAMESPEED_ONLINE


## Results
### 2. Victory timing and the settlement balance

In [2]:
for age in analyze.AGES:
    end = summary["ages"][age]["last"]
    print(age, "turn",end["turn"],"cities/towns",int(end["cities"]),int(end["towns"]),
          "Innovation",summary["ages"][age]["innovation_changes"][-1]["innovation"])
print("Rocketry available:",summary["ages"]["MODERN"]["gate_open"])
print("Modern Innovation changes:")
for row in summary["ages"]["MODERN"]["innovation_changes"]:
    print(row)
modern_initiative = [r for r in builds if r["age"]=="MODERN" and r["player"]==0
                     and r["item"]=="PROJECT_CITY_SCIENCE_PROJECT"]
assert not modern_initiative
print("Modern Science Initiative city-turns:",len(modern_initiative))


ANTIQUITY turn 70 cities/towns 3 3 Innovation 18
EXPLORATION turn 70 cities/towns 6 4 Innovation 47
MODERN turn 93 cities/towns 6 10 Innovation 123
Rocketry available: 51
Modern Innovation changes:
{'turn': 1, 'innovation': 47, 'rival': 23, 'countdown': None}
{'turn': 51, 'innovation': 52, 'rival': 25, 'countdown': None}
{'turn': 55, 'innovation': 57, 'rival': 25, 'countdown': None}
{'turn': 60, 'innovation': 59, 'rival': 25, 'countdown': None}
{'turn': 61, 'innovation': 61, 'rival': 25, 'countdown': None}
{'turn': 62, 'innovation': 63, 'rival': 25, 'countdown': None}
{'turn': 63, 'innovation': 65, 'rival': 25, 'countdown': None}
{'turn': 65, 'innovation': 67, 'rival': 27, 'countdown': None}
{'turn': 71, 'innovation': 70, 'rival': 27, 'countdown': None}
{'turn': 82, 'innovation': 73, 'rival': 31, 'countdown': None}
{'turn': 84, 'innovation': 88, 'rival': 31, 'countdown': None}
{'turn': 88, 'innovation': 118, 'rival': 31, 'countdown': 5}
{'turn': 90, 'innovation': 123, 'rival': 31, 'cou

### 3. Local happiness and the policy-change interval

In [3]:
happiness = analyze.happiness_summary(city_turns)
print("City | negative turns | minimum happiness | turns at 80% cap | mean production")
for row in happiness:
    print(row["city"],row["negative_turns"],row["minimum_happiness"],
          row["turns_at_penalty_cap"],row["mean_production"],sep=" | ")
print("Core settlement output around policy changes:")
for turn in [16,17,40,41,42,43]:
    sample=[r for r in city_turns if r["turn"]==turn and r["name"] in analyze.CORE_CITIES]
    print(turn,"Production",sum(r["Production"] for r in sample),
          "Science",sum(r["Science"] for r in sample),"Happiness",sum(r["Happiness"] for r in sample))
print("Fixed-state, zero-unhappiness potential recovery (upper bounds, not forecasts):")
print("Production",sum(r["potential_production_recovery"] for r in happiness),
      "local Science",sum(r["potential_science_recovery"] for r in happiness))


City | negative turns | minimum happiness | turns at 80% cap | mean production
Ниневия | 81 | -26 | 16 | 40.5
Кальху | 93 | -33 | 29 | 22.9
Арби-Илу | 84 | -19 | 13 | 38.4
Аррапха | 85 | -11 | 0 | 45.6
Шираз | 0 | 2 | 0 | 41.8
Ахваз | 72 | -10 | 0 | 27.1
Core settlement output around policy changes:
16 Production 187 Science 260 Happiness -22
17 Production 110 Science 182 Happiness -67
40 Production 156 Science 261 Happiness -68
41 Production 187 Science 315 Happiness -39
42 Production 269 Science 471 Happiness 5
43 Production 227 Science 406 Happiness -22
Fixed-state, zero-unhappiness potential recovery (upper bounds, not forecasts):
Production 8209 local Science 15574


### 4. Treasury and production allocation

In [4]:
finance = analyze.modern_finance(stats,frames)
print(json.dumps(finance,ensure_ascii=False,indent=2))
from collections import Counter
allocation=Counter()
for row in builds:
    if row["age"]!="MODERN" or row["player"]!=0:
        continue
    item=row["item"]
    if item=="UNIT_SETTLER": kind="Settlers"
    elif item.startswith("UNIT_"): kind="Other units"
    elif "FORTIFICATION" in item: kind="Walls"
    elif item in ["BUILDING_LAUNCH_PAD","BUILDING_AIRFIELD"] or item.startswith("PROJECT_"): kind="Space work"
    else: kind="Other buildings"
    allocation[kind]+=1
assert sum(allocation.values())==552
print("Modern city-turns:",dict(allocation))
print("Completed Launch Pads and Preparation runs:")
for row in runs:
    if row["age"]=="MODERN" and row["player"]==0 and row["completed"] and row["item"] in ["BUILDING_LAUNCH_PAD","PROJECT_CREWED_SPACE_FLIGHT_PREP"]:
        print(row["city"],row["item"],row["start"],row["end"])


{
  "maximum_balance": 1557.4,
  "zero_balance_turns": [
    80,
    81,
    82
  ],
  "negative_income_turns": [
    17,
    66,
    67,
    68,
    69,
    70,
    71,
    72,
    73,
    74,
    76,
    77,
    79,
    80,
    81,
    82,
    88
  ],
  "recorded_spending": {
    "GARRISON": 700,
    "WALLS": 1068,
    "LEVER": 2520,
    "REPAIR": 1080,
    "DEFENDER": 2680,
    "UPGRADE": 1920
  },
  "goal_turns": {
    "PADS": 5,
    "none": 85,
    "LEVER": 1,
    "AIRFIELD": 1
  }
}
Modern city-turns: {'Settlers': 30, 'Other buildings': 257, 'Other units': 47, 'Walls': 36, 'Space work': 182}
Completed Launch Pads and Preparation runs:
LOC_CITY_NAME_ASSYRIA1 BUILDING_LAUNCH_PAD 67 73
LOC_CITY_NAME_ASSYRIA1 PROJECT_CREWED_SPACE_FLIGHT_PREP 75 88
LOC_CITY_NAME_ASSYRIA11 BUILDING_LAUNCH_PAD 54 68
LOC_CITY_NAME_ASSYRIA11 PROJECT_CREWED_SPACE_FLIGHT_PREP 70 84
LOC_CITY_NAME_ASSYRIA4 BUILDING_LAUNCH_PAD 62 75
LOC_CITY_NAME_ASSYRIA4 PROJECT_CREWED_SPACE_FLIGHT_PREP 76 88
LOC_CITY_NAME_AB

## Takeaways
1. Price local unhappiness correctly in policy replacements, buildings and specialist decisions.
2. Recover the six core cities before buying more production queues.
3. Compare production upgrades and happiness recovery against complete time to the science goal.
4. Evaluate late founding and military obligations against the science race.
5. Trial higher Production/Happiness and a post-Rocketry reduction in Science weight on the same seed. The proposed preset is not proven optimal.

This notebook uses only the Python standard library and its adjacent helper/archive. Its code cells were compiled and executed top-to-bottom from a fresh namespace; no Jupyter-only features are needed. The JSON structure, execution counts, error-free outputs and input uniqueness checks were validated.
